# AI Market Intelligence — EDA & Methodology Walkthrough

This notebook walks through the data-science workflow behind the platform: data collection, exploratory analysis, stationarity checks, leakage-aware model validation (including a naive baseline), model interpretability, and honest treatment of forecast uncertainty.

Run the cells from top to bottom. Keep Jupyter started from the **project root** so the `app/` package is importable.


## 1. Setup
We reuse the platform's own ingestion and analytics modules so this notebook always reflects the live code.


In [ ]:
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from app.services.market_data import fetch_price_history, fetch_fear_greed_history
from app.services.analytics import analyze_history, _build_features

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


## 2. Data collection
`fetch_price_history` calls CoinGecko and caches the response on disk, so repeated runs do not hit public rate limits. The Fear & Greed Index is merged as an exogenous sentiment feature.


In [ ]:
COIN = "bitcoin"
DAYS = 90

history = fetch_price_history(COIN, DAYS)
fear_greed = fetch_fear_greed_history(DAYS)
print(f"price points: {len(history['prices'])} · fear & greed records: {len(fear_greed)}")

frame, candles = _build_features(history, fear_greed)
frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True)
frame.head()


## 3. Exploratory data analysis
Price levels are far more useful when we look at **returns** (percentage changes), which are comparable across time and assets. Returns also matter because most regressors struggle with non-stationary levels.


In [ ]:
frame["return_pct"] = frame["price_usd"].pct_change() * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=frame["timestamp"], y=frame["price_usd"], name="Price (USD)", line=dict(color="#1565C0")))
fig.add_trace(go.Scatter(x=frame["timestamp"], y=frame["sma_7"], name="SMA 7", line=dict(color="#C69214")))
fig.add_trace(go.Scatter(x=frame["timestamp"], y=frame["sma_30"], name="SMA 30", line=dict(color="#6A1B9A")))
fig.update_layout(height=380, yaxis_title="USD", legend=dict(orientation="h"))
fig.show()


In [ ]:
fig = px.histogram(frame["return_pct"].dropna(), nbins=60, title="Distribution of daily returns (%)")
fig.update_layout(height=320, xaxis_title="Daily return (%)")
fig.show()

print(f"mean return: {frame['return_pct'].mean():+.3f}% · std: {frame['return_pct'].std():.3f}% · skew: {frame['return_pct'].skew():+.3f}")


## 4. Stationarity (ADF test)
A unit-root (non-stationary) series can produce spurious regressions. We run an Augmented Dickey-Fuller test on the **differenced** series to confirm the platform's assumption that first differences are stationary.


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_pvalue = adfuller(frame["price_usd"].diff().dropna(), autolag="AIC")[1]
print(f"ADF p-value on first differences: {adf_pvalue:.6f}")
print("Stationary after differencing:", adf_pvalue < 0.05)


## 5. Feature engineering
Features used by the models:
- **Lagged prices** (`lag_1`, `lag_2`, `lag_7`) - short-term momentum.
- **Moving averages** (`sma_7`, `sma_30`) - trend state.
- **RSI(14)** - momentum / mean-reversion pressure.
- **VWAP** and **volume** - participation and liquidity.
- **Fear & Greed Index** - market sentiment, aligned by date.

Targets are forward-looking (`target_next_price`, `next_return`), so every model predicts *future* values.


In [ ]:
frame[["timestamp", "price_usd", "lag_1", "sma_7", "rsi_14", "vwap", "fear_greed"]].tail(10)


## 6. Leakage-aware validation
A random train/test split would leak the future into training and give optimistic results. The platform uses **TimeSeriesSplit**: each fold trains only on the past and evaluates on a strictly future window.


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

X = np.arange(len(frame))
tscv = TimeSeriesSplit(n_splits=3)
fig = go.Figure()
colors = ["#1565C0", "#C69214", "#2E7D32"]
for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    fig.add_trace(go.Scatter(x=train_idx, y=[fold + 1] * len(train_idx), mode="markers", marker=dict(color=colors[fold], size=4), name=f"Fold {fold + 1} train"))
    fig.add_trace(go.Scatter(x=test_idx, y=[fold + 1] * len(test_idx), mode="markers", marker=dict(color="#D32F2F", size=4), name=f"Fold {fold + 1} test", showlegend=False))
fig.update_layout(height=220, yaxis=dict(tickvals=[1, 2, 3], title="Fold"), xaxis_title="Time index", title="TimeSeriesSplit - test windows are always in the future")
fig.show()


## 7. Model comparison vs a naive baseline
Every model is scored only on held-out future folds. A **persistence (naive)** baseline - predicting the last known price - is included so we can ask the honest question: *does the model beat doing nothing?*


In [ ]:
analysis = analyze_history(history, fear_greed, forecast_days=30)
comparison = pd.DataFrame(analysis["model_comparison"]).sort_values("rmse")
comparison["mae"] = comparison["mae"].map("${:,.0f}".format)
comparison["rmse"] = comparison["rmse"].map("${:,.0f}".format)
comparison


### Interpretation
On the 90-day Bitcoin window the naive baseline is the hardest to beat - a classic sign that crypto levels are close to a random walk. That is a real finding, not a bug: it is exactly why naive baselines belong in every comparison.


## 8. Interpretability - feature importance
The dashboard shows gain-based importances from XGBoost. They say *which inputs moved the forecast most*, making the model explainable without a paid LLM.


In [ ]:
importance = pd.DataFrame(analysis["feature_importance"])
fig = px.bar(importance.sort_values("importance"), x="importance", y="feature", orientation="h", color="importance", color_continuous_scale="Blues", labels={"importance": "Gain", "feature": "Feature"})
fig.update_layout(height=300, yaxis=dict(autorange="reversed"), coloraxis_showscale=False)
fig.show()


## 9. Forecast & uncertainty
The 95% prediction interval **widens with the horizon** (`sqrt(h)` growth) because uncertainty accumulates the further we forecast. For horizons beyond 90 days the platform switches from recursive tree/linear forecasts to an ARIMA baseline, since recursive forecasts compound their own errors.


In [ ]:
forecast = pd.DataFrame(analysis["forecast"])
forecast["timestamp"] = pd.to_datetime(forecast["timestamp"])

fig = go.Figure()
fig.add_trace(go.Scatter(x=frame["timestamp"], y=frame["price_usd"], name="Actual", line=dict(color="#1565C0", width=2)))
fig.add_trace(go.Scatter(x=forecast["timestamp"], y=forecast["lower_95"], name="95% lower", line=dict(width=0)))
fig.add_trace(go.Scatter(x=forecast["timestamp"], y=forecast["upper_95"], name="95% interval", line=dict(width=0), fill="tonexty", fillcolor="rgba(106,27,154,.13)"))
fig.add_trace(go.Scatter(x=forecast["timestamp"], y=forecast["price_usd"], name="Forecast", line=dict(color="#6A1B9A", dash="dash", width=2)))
fig.update_layout(height=360, yaxis_title="USD", legend=dict(orientation="h"), title=f"Forecast - {analysis['metrics']['selected_model']}")
fig.show()


## 10. Signal backtest
The model's BUY/HOLD signals are backtested against a simple buy-and-hold strategy on the same future folds - a transparent, apples-to-apples comparison.


In [ ]:
signals = pd.DataFrame(analysis["signals"])
if not signals.empty:
    strategy = (1 + signals["strategy_return"].fillna(0)).prod() - 1
    buy_hold = (1 + signals["actual_return"].fillna(0)).prod() - 1
    print(f"Model signal return : {strategy * 100:+.2f}%")
    print(f"Buy & hold return   : {buy_hold * 100:+.2f}%")
else:
    print("No backtest signals (need more history).")


## 11. Limitations & honest takeaways

- **Not a trading system.** Forecasts are educational baselines with wide uncertainty; never make financial decisions from them.
- **Short history.** A few months of data cannot support 1-3 year forecasts; long horizons are scenario extrapolations.
- **Naive baseline wins.** If the ML models cannot consistently beat persistence, the right move is to say so and focus on better features and regimes rather than fancier models.
- **Natural next steps:** hyperparameter tuning (GridSearchCV over TimeSeriesSplit), regime detection, richer macro/sentiment features, and probabilistic calibration of the interval.


In [ ]:
print("Notebook complete.")
